# P14B — Perbandingan protokol dan baseline

Random split dan group holdout menjawab sasaran generalisasi berbeda. Keluaran hanya menjelaskan generator sintetis.

In [1]:
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.model_selection import train_test_split,GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
SEED=14014;df=pd.read_csv('Bab14_Paket/Data/slope_campaign_synthetic.csv')
features=['slope_angle_deg_synth','cohesion_kPa_synth','drainage_score_synth','vegetation_cover_pct_synth','rain_7d_mm_synth','pore_pressure_index_synth','soil_state_index_synth','campaign_index'];target='deformation_response_synth_mm';X=df[features];y=df[target]
tr,te=train_test_split(np.arange(len(df)),test_size=.25,random_state=SEED)
gss=GroupShuffleSplit(n_splits=1,test_size=.25,random_state=SEED);tg,eg=next(gss.split(X,y,groups=df.slope_id))
def score(name,a,b):
 baseline=np.repeat(y.iloc[a].mean(),len(b));m=RandomForestRegressor(n_estimators=250,min_samples_leaf=3,max_features=.8,random_state=SEED,n_jobs=-1);m.fit(X.iloc[a],y.iloc[a]);pred=m.predict(X.iloc[b]);return [name,len(a),len(b),df.iloc[a].slope_id.nunique(),df.iloc[b].slope_id.nunique(),mean_absolute_error(y.iloc[b],baseline),mean_absolute_error(y.iloc[b],pred),mean_squared_error(y.iloc[b],pred)**.5,r2_score(y.iloc[b],pred)]
r=pd.DataFrame([score('random_row',tr,te),score('group_holdout_slope',tg,eg)],columns=['split','n_train','n_test','groups_train','groups_test','baseline_MAE','model_MAE','model_RMSE','model_R2']);r.to_csv('Bab14_Paket/Data/P14_protocol_demo_metrics.csv',index=False);print(r.round(4).to_string(index=False));print('Synthetic DGP recovery only; not field performance.')

              split  n_train  n_test  groups_train  groups_test  baseline_MAE  model_MAE  model_RMSE  model_R2
         random_row      972     324            72           72        0.7280     0.0901      0.1299    0.9764
group_holdout_slope      972     324            54           18        0.7035     0.7051      0.8400   -0.8323
Synthetic DGP recovery only; not field performance.
